In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import MetaTrader5 as mt5
import datetime
import plotly.graph_objects as go
from scipy.stats import t

In [2]:
if not mt5.initialize():
    print("MT5 Init Failed")
    quit()
utc_from = datetime.datetime(2010, 1, 1, tzinfo=datetime.timezone.utc) 
utc_to = datetime.datetime(2025, 12, 31, tzinfo=datetime.timezone.utc)
timeframe = mt5.TIMEFRAME_D1
us_oil_data = mt5.copy_rates_range('USOIL', timeframe, utc_from, utc_to)
uk_oil_data = mt5.copy_rates_range('UKOIL', timeframe, utc_from, utc_to)
mt5.shutdown()
if us_oil_data is None or uk_oil_data is None:
    print("No data retrieved.")
    quit()

df_us = pd.DataFrame(us_oil_data)
df_uk = pd.DataFrame(uk_oil_data)
df_us['time'] = pd.to_datetime(df_us['time'], unit='s')
df_us.set_index('time', inplace=True)
df_uk['time'] = pd.to_datetime(df_uk['time'], unit='s')
df_uk.set_index('time', inplace=True)

In [3]:
df_combined = pd.concat([df_us['close'], df_uk['close']], axis=1, keys=['us_close', 'uk_close'])
df_combined.dropna(inplace=True)
df_combined['spread'] = df_combined['uk_close'] - df_combined['us_close']
df_combined['spread_us_pct'] = df_combined['spread']/df_combined['us_close']

In [4]:
# 1. Get Data
data = df_combined['spread_us_pct']

# 2. Fit t-distribution
df_params, loc, scale = t.fit(data)

# 3. Create Histogram
fig = px.histogram(data, nbins=100, 
                   histnorm='probability density', 
                   title=f"WTI-Brent spread Risk Model (Mean: {loc:.4f})",
                   opacity=0.6)

# 4. Generate Curve
x_range = np.linspace(data.min(), data.max(), 1000)
pdf_values = t.pdf(x_range, df_params, loc, scale)
fig.add_trace(go.Scatter(x=x_range, y=pdf_values, mode='lines', name='T-Dist Fit', line=dict(color='red', width=3)))

# 5. CORRECT CALCULATION for 95% and 99% Intervals
# We want the bounds that contain 95% of the data (leaving 2.5% on each tail)
# We want the bounds that contain 99% of the data (leaving 0.5% on each tail)

# 95% Confidence Interval (2.5% percentile and 97.5% percentile)
lower_95 = t.ppf(0.025, df_params, loc, scale)
upper_95 = t.ppf(0.975, df_params, loc, scale)

# 99% Confidence Interval (0.5% percentile and 99.5% percentile)
lower_99 = t.ppf(0.005, df_params, loc, scale)
upper_99 = t.ppf(0.995, df_params, loc, scale)

# 6. Add Lines (Using the EXACT calculated values)
# 95% Lines (Orange)
fig.add_vline(x=upper_95, line_dash="dash", line_color="orange", annotation_text="95% Upper")
fig.add_vline(x=lower_95, line_dash="dash", line_color="orange", annotation_text="95% Lower")

# 99% Lines (Red - Stop Loss Zones)
fig.add_vline(x=upper_99, line_dash="dot", line_color="red", annotation_text="99% Upper")
fig.add_vline(x=lower_99, line_dash="dot", line_color="red", annotation_text="99% Lower")

# Mean Line (Black)
fig.add_vline(x=loc, line_width=2, line_color="black", annotation_text="Mean")

fig.show()

In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 1. SETUP & DATA GENERATION (Replace with your actual data)
# ==========================================
# For this example, I'll create dummy data to show the logic works.
# In your real code, just load your df_combined from earlier.
# np.random.seed(42)
# dates = pd.date_range(start='2023-01-01', periods=5000, freq='H')
# us_oil = 70 + np.cumsum(np.random.normal(0, 0.5, 5000))
# uk_oil = us_oil + 5 + np.random.normal(0, 0.8, 5000) # Correlated but noisy

# df = pd.DataFrame({'USOIL': us_oil, 'UKOIL': uk_oil}, index=dates)
df = df_combined.copy()

# ==========================================
# 2. STRATEGY PARAMETERS
# ==========================================
LOOKBACK_WINDOW = 168  # 1 Week of hourly data (24 * 7) - Typical for swing trading
ENTRY_THRESHOLD = 2.0  # Enter when spread is 2 sigma away
EXIT_THRESHOLD = 0.0   # Exit when spread returns to mean
STOP_LOSS = 4.0        # Close if spread blows out (Risk Management)

# ==========================================
# 3. CALCULATE INDICATORS
# ==========================================
# A. Calculate the spread
# Simple diff is okay, but Log spread is better for long-term (pct difference)
# df['spread'] = np.log(df['USOIL']) - np.log(df['UKOIL'])

# B. Calculate ROLLING Stats
df['Roll_Mean'] = df['spread'].rolling(window=LOOKBACK_WINDOW).mean()
df['Roll_Std'] = df['spread'].rolling(window=LOOKBACK_WINDOW).std()

# C. Calculate Z-Score
df['Z_Score'] = (df['spread'] - df['Roll_Mean']) / df['Roll_Std']

# Drop NaN values from the warmup period
df.dropna(inplace=True)

# ==========================================
# 4. GENERATE SIGNALS (Vectorized Logic)
# ==========================================
df['Position'] = 0 # 1 = Long spread (Buy US, Sell UK), -1 = Short spread

# We use a loop for state management (Entry vs Exit logic is hard to vectorize perfectly)
current_pos = 0

position_history = []

for z in df['Z_Score']:
    # ENTRY LOGIC
    if current_pos == 0:
        if z < -ENTRY_THRESHOLD: 
            current_pos = 1  # Long the spread (WTI is cheap)
        elif z > ENTRY_THRESHOLD:
            current_pos = -1 # Short the spread (WTI is expensive)
            
    # EXIT LOGIC (Mean Reversion)
    elif current_pos == 1:
        if z >= EXIT_THRESHOLD: # Reverted to mean
            current_pos = 0
        elif z < -STOP_LOSS:    # Stop Loss (spread widening too much)
            current_pos = 0
            
    elif current_pos == -1:
        if z <= -EXIT_THRESHOLD: # Reverted to mean
            current_pos = 0
        elif z > STOP_LOSS:      # Stop Loss
            current_pos = 0
            
    position_history.append(current_pos)

df['Position'] = position_history

# ==========================================
# 5. CALCULATE RETURNS
# ==========================================
# PnL = Position * Change in spread
# Note: In real life, PnL is (Change in US) - (Change in UK)
# Since we used log spread, diff in spread approx returns.
df['Strategy_Returns'] = df['Position'].shift(1) * df['spread'].diff()

# Cumulative Returns
df['Cum_Returns'] = df['Strategy_Returns'].cumsum()

# ==========================================
# 6. VISUALIZE (The "Calgary Dashboard")
# ==========================================
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.05, 
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=("Cumulative PnL", "Rolling Z-Score", "spread"))

# Plot 1: Equity Curve
fig.add_trace(go.Scatter(x=df.index, y=df['Cum_Returns'], name="Strategy PnL", line=dict(color='green')), row=1, col=1)

# Plot 2: Z-Score with Thresholds
fig.add_trace(go.Scatter(x=df.index, y=df['Z_Score'], name="Z-Score", line=dict(color='blue', width=1)), row=2, col=1)
fig.add_hline(y=ENTRY_THRESHOLD, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=-ENTRY_THRESHOLD, line_dash="dash", line_color="green", row=2, col=1)
fig.add_hline(y=0, line_color="black", row=2, col=1)

# Plot 3: The Actual spread
fig.add_trace(go.Scatter(x=df.index, y=df['spread'], name="spread", line=dict(color='orange')), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['Roll_Mean'], name="Roll Mean", line=dict(color='black', dash='dot')), row=3, col=1)

fig.update_layout(title="Rolling Mean Reversion Strategy (USOIL vs UKOIL)", height=900)
fig.show()

# ==========================================
# 7. METRICS (For your Resume)
# ==========================================
total_return = df['Cum_Returns'].iloc[-1]
sharpe = (df['Strategy_Returns'].mean() / df['Strategy_Returns'].std()) * np.sqrt(252 * 24) # Annualized (Hourly data)

print(f"Total Return: {total_return:.4f} Log Units")
print(f"Sharpe Ratio: {sharpe:.2f}")

Total Return: 12.6150 Log Units
Sharpe Ratio: 3.56
